# TechArena CTR: 64-dimensional full-prefix and warm-click auxiliary model
Attach the private TechArena Neural Causal Histories dataset. GPU required.
Fixed 20 epochs, all pre-cutoff events, sampled warm-history objective; no validation-label fitting.
Local target 0.83 remains unachieved until measured.


In [ ]:
from pathlib import Path
import importlib.util
import torch
roots = list(Path("/kaggle/input").rglob("neural_manifest.json"))
assert len(roots) == 1, "Attach TechArena Neural Causal Histories"
NEURAL_ROOT = roots[0].parent
source_path = Path("/kaggle/working/neural_aux_runner.py")
source_path.write_text("\"\"\"Full-prefix joint embeddings with auxiliary pre-window click learning.\n\nThe only held-out labels used are for final scoring. No epoch, blend weight,\nor recipe selection uses validation labels. This is exploratory on reused dates.\n\"\"\"\nimport importlib.util\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nRECIPE = dict(seed=6502, dim=64, epochs=20, batch_size=2048, learning_rate=.001,\n              weight_decay=.0001, anchor_penalty=.005, auxiliary_weight=1.,\n              rank_residual_weight=.1, negatives=10, training=\"all events before cutoff\")\n\n\ndef load_module(name, path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    m=importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(m)\n    return m\n\n\ndef expand(factors, context, rng, dim):\n    old = factors[\"V\"].shape[1]\n    assert dim >= old\n    values = {k:v.copy() for k,v in factors.items()}\n    for k in [\"U\",\"V\",\"H\"]:\n        extra=np.zeros((len(values[k]),dim-old),dtype=\"float32\")\n        if k==\"V\":extra[:]=rng.normal(0,.05,extra.shape)\n        values[k]=np.column_stack([values[k],extra])\n    matrices={}\n    for name, v in context.items():\n        a=torch.zeros((*v.shape[:-2],dim,dim),dtype=v.dtype)\n        a[..., :old, :old]=v\n        matrices[name]=a\n    return values,matrices\n\n\ndef warm_pairs(data,fitting,n_items):\n    # First pre-cutoff event per user, including supplied warm-up history.\n    _,first=np.unique(data[\"users\"][fitting],return_index=True)\n    ix=fitting[first]\n    h=data[\"hist\"][ix];valid=h>=0\n    users=np.repeat(data[\"users\"][ix],h.shape[1])[valid.ravel()]\n    items=h.ravel()[valid.ravel()]\n    gaps=data[\"gaps\"][ix].ravel()[valid.ravel()]\n    keys=np.rec.fromarrays([users,items,gaps],names=\"user,item,gap\")\n    _,unique=np.unique(keys,return_index=True)\n    users,items=users[unique],items[unique]\n    counts=np.bincount(items,minlength=n_items)+1\n    return users,items,np.log(counts/counts.mean()).astype(\"float32\")\n\n\ndef fit(root, output, day, data, manifest, base, metrics, device):\n    started=time.monotonic()\n    rng=np.random.default_rng(RECIPE[\"seed\"])\n    torch.manual_seed(RECIPE[\"seed\"])\n    torch.cuda.manual_seed_all(RECIPE[\"seed\"])\n    output.mkdir(parents=True,exist_ok=False)\n    cut=manifest[\"start\"]+day*86400\n    fitting=np.flatnonzero(data[\"timestamp\"]<cut)\n    val=dict(np.load(root/f\"val{day}.npz\"));vi=val[\"indices\"]\n    assert data[\"timestamp\"][fitting].max()<data[\"timestamp\"][vi].min()\n    assert not np.intersect1d(fitting,vi).size\n    with np.load(root/f\"factors{day}.npz\") as z:\n        factors={k:z[k].astype(\"float32\") for k in [\"U\",\"V\",\"H\",\"bu\",\"bi\"]}\n    context=torch.load(root/f\"context{day}.pt\",weights_only=True,map_location=\"cpu\")\n    factors,context=expand(factors,context,rng,RECIPE[\"dim\"])\n    model=base.JointModel(factors,context).to(device)\n    initial=base.predict(model,data,vi,device)\n    expected=np.load(root/f\"initial{day}.npy\")\n    error=float(np.max(np.abs(initial-expected)))\n    assert error<3e-6,error\n    wu,wi,pop_bias=warm_pairs(data,fitting,len(factors[\"V\"]))\n    assert len(wu)>0\n    pop=torch.as_tensor(pop_bias,device=device)\n    anchors={k:p.detach().clone() for k,p in model.named_parameters()}\n    optimizer=torch.optim.AdamW(model.parameters(),lr=RECIPE[\"learning_rate\"],weight_decay=RECIPE[\"weight_decay\"])\n    recipe=dict(RECIPE,day=day,cutoff=cut,fit_events=len(fitting),warm_pairs=len(wu),\n                last_fit_timestamp=int(data[\"timestamp\"][fitting].max()),initial_replay_error=error,\n                input_manifest_sha256=base.digest(root/\"neural_manifest.json\"),\n                source_sha256=base.digest(Path(__file__)),torch=torch.__version__)\n    (output/\"recipe.json\").write_text(json.dumps(recipe,indent=2))\n    print(dict(stage=\"prepared\",day=day,fit_events=len(fitting),warm_pairs=len(wu),initial_replay_error=error),flush=True)\n    for epoch in range(1,RECIPE[\"epochs\"]+1):\n        model.train();order=rng.permutation(fitting);loss_total=0.;warm_total=0.\n        for off in range(0,len(order),RECIPE[\"batch_size\"]):\n            ix=order[off:off+RECIPE[\"batch_size\"]]\n            optimizer.zero_grad(set_to_none=True)\n            logits=model(*base.batch(data,ix,device))\n            loss=nn.functional.cross_entropy(logits,torch.as_tensor(data[\"target\"][ix],device=device,dtype=torch.long))\n            chosen=rng.integers(0,len(wu),size=len(ix))\n            users=torch.as_tensor(wu[chosen],device=device,dtype=torch.long)\n            positives=wi[chosen]\n            negative=rng.integers(0,len(pop_bias),size=(len(ix),RECIPE[\"negatives\"]),dtype=np.int64)\n            same=negative==positives[:,None]\n            while same.any():\n                negative[same]=rng.integers(0,len(pop_bias),size=int(same.sum()))\n                same=negative==positives[:,None]\n            candidates=torch.as_tensor(np.column_stack([positives,negative]),device=device,dtype=torch.long)\n            scores=(model.U(users)[:,None]*model.V(candidates)).sum(-1)+pop[candidates]\n            warm_loss=nn.functional.cross_entropy(scores,torch.zeros(len(ix),device=device,dtype=torch.long))\n            drift=sum((p-anchors[name]).square().mean() for name,p in model.named_parameters())\n            total=loss+RECIPE[\"auxiliary_weight\"]*warm_loss+RECIPE[\"anchor_penalty\"]*drift\n            assert torch.isfinite(total), \"Nonfinite training loss\"\n            total.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);optimizer.step()\n            loss_total+=float(loss.detach())*len(ix);warm_total+=float(warm_loss.detach())*len(ix)\n        p=dict(day=day,epoch=epoch,epochs=RECIPE[\"epochs\"],event_loss=loss_total/len(order),\n               auxiliary_loss=warm_total/len(order),elapsed_s=round(time.monotonic()-started,1))\n        (output/\"progress.json\").write_text(json.dumps(p));print(p,flush=True)\n    torch.save(model.state_dict(),output/\"model.pt\")\n    raw=base.predict(model,data,vi,device)\n    pred=metrics.rank(metrics.rank(val[\"baseline\"])+RECIPE[\"rank_residual_weight\"]*(metrics.rank(raw)-metrics.rank(initial)))\n    m=metrics.metrics(val[\"y\"],pred,val[\"baseline\"],val[\"weights\"])\n    m.update(day=day,raw_auc=float(metrics.roc_auc_score(val[\"y\"],raw)),initial_raw_auc=float(metrics.roc_auc_score(val[\"y\"],initial)),\n             initial_replay_error=error,model_sha256=base.digest(output/\"model.pt\"),official_auc=None,csv_ready=False)\n    restored=base.JointModel(factors,context).to(device)\n    restored.load_state_dict(torch.load(output/\"model.pt\",weights_only=True,map_location=device))\n    replay=base.predict(restored,data,vi,device)\n    assert np.array_equal(replay,raw), \"Saved checkpoint prediction changed\"\n    m[\"saved_checkpoint_replay\"]=\"PASS\"\n    np.savez_compressed(output/\"predictions.npz\",raw=raw,initial=initial,pred=pred,**val)\n    (output/\"report.json\").write_text(json.dumps(m,indent=2));print(json.dumps(m,indent=2),flush=True)\n    return m\n\n\ndef run(root,output,device=\"cuda\"):\n    root,output=Path(root),Path(output)\n    assert torch.cuda.is_available() if device==\"cuda\" else True\n    torch.set_num_threads(4)\n    base=load_module(\"joint_base\",root/\"neural_runner.py\")\n    metrics=load_module(\"joint_metrics\",root/\"runner.py\")\n    manifest=json.loads((root/\"neural_manifest.json\").read_text())\n    for name,info in manifest[\"files\"].items():\n        assert base.digest(root/name)==info[\"sha256\"],name\n    output.mkdir(parents=True,exist_ok=False)\n    (output/\"neural_aux_runner.py\").write_text(Path(__file__).read_text())\n    data=dict(np.load(root/\"events.npz\"))\n    reports=[fit(root,output/\"day6\",6,data,manifest,base,metrics,device)]\n    if reports[0][\"positive_gate\"]:\n        reports.append(fit(root,output/\"day5\",5,data,manifest,base,metrics,device))\n    summary=dict(reports=reports,eligible_for_review=len(reports)==2 and all(m[\"positive_gate\"] for m in reports),\n                 target=.83,target_achieved=False,csv_ready=False,official_auc=None,\n                 note=\"Full-prefix training, pre-cutoff warm histories only. Reused dates; no official forecast.\")\n    (output/\"summary.json\").write_text(json.dumps(summary,indent=2))\n    return summary\n", encoding="utf-8")
import hashlib
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == "6e327251396af9c1e4d00a2a585d7eb14fe4cacac99fdc69d47616897f22030c"
spec = importlib.util.spec_from_file_location("ctr_aux_neural", source_path)
aux = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aux)
RUN_NAME = "ctr-neural-002-aux64"
output = Path("/kaggle/working") / RUN_NAME
print("CUDA available:", torch.cuda.is_available())
print("Full-prefix plus warm-click auxiliary training", aux.RECIPE)
summary = aux.run(NEURAL_ROOT, output, device="cuda")
summary


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import zipfile
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
print("Neural diagnostic results; no competition CSV or official score.")
